# Example: Join Staff Injury Rates to CMS Nursing Home Quality Data

A short worked example of how to use this repository's published files. It reads three files over the network, joins them, and asks one question: **do nursing homes with more nursing staff per resident have fewer staff handling injuries?**

Nothing is downloaded from OSHA and no classification is re-run. The whole example runs in under a minute.

**Files used**
- `injury-rates/derived/facility-rates-2025.csv` — one row per establishment, with handling-injury counts, hours, and rates
- `crosswalk/osha-cms-crosswalk-nf-2025.csv` — OSHA establishment ID to CMS Certification Number
- CMS *Provider Information* (nursing homes), fetched from the CMS catalog at run time

**Source data** US Government work, 17 U.S.C. §105

## 1. Load the repository's published files

In [ ]:
import pandas as pd, numpy as np, requests
from scipy import stats
BASE = 'https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/main/'
fac = pd.read_csv(BASE + 'injury-rates/derived/facility-rates-2025.csv', dtype={'establishment_id': str, 'cms_ccn': str})
nf = fac[(fac.setting == 'nursing facility') & fac.cms_ccn.notna()].copy()
print(f'{len(fac):,} establishments with injury rates; {len(nf):,} nursing facilities carry a CMS Certification Number')
nf.head(3)

## 2. Fetch the current CMS Provider Information file

CMS changes the file's URL every month; the catalog does not.

In [ ]:
catalog = requests.get('https://data.cms.gov/provider-data/data.json', timeout=60).json()['dataset']
d = next(x for x in catalog if x['title'].strip().lower() == 'provider information'
         and 'nursing' in (' '.join(x.get('theme', [])) + x.get('description', '')).lower())
url = next(x['downloadURL'] for x in d['distribution'] if x.get('downloadURL', '').endswith('.csv'))
print(f"{d['title']} | modified {d.get('modified')}")
cms = pd.read_csv(url, dtype=str, low_memory=False)
col = {c.lower(): c for c in cms.columns}
g = lambda *n: next(col[x.lower()] for x in n if x.lower() in col)
cms = cms.rename(columns={
    g('CMS Certification Number (CCN)', 'Federal Provider Number'): 'ccn',
    g('Overall Rating'): 'overall_rating',
    g('Reported Total Nurse Staffing Hours per Resident per Day'): 'nurse_hours_per_resident_day',
    g('Ownership Type'): 'ownership',
    g('Number of Certified Beds'): 'beds'})
cms = cms[['ccn', 'overall_rating', 'nurse_hours_per_resident_day', 'ownership', 'beds']]
for c in ['overall_rating', 'nurse_hours_per_resident_day', 'beds']:
    cms[c] = pd.to_numeric(cms[c], errors='coerce')
print(f'{len(cms):,} CMS nursing homes')

## 3. Join and look at the question

In [ ]:
a = nf.merge(cms, left_on='cms_ccn', right_on='ccn', how='inner')
print(f'{len(a):,} nursing homes with both a staff handling-injury rate and CMS data')

a['staffing_quintile'] = pd.qcut(a.nurse_hours_per_resident_day, 5, labels=['1 lowest', '2', '3', '4', '5 highest'])
by_staffing = a.groupby('staffing_quintile', observed=True).agg(
    facilities=('handling_rate', 'size'),
    median_nurse_hours=('nurse_hours_per_resident_day', 'median'),
    median_handling_rate=('handling_rate', 'median'),
    pooled_handling_rate=('handling', lambda s: round(s.sum() * 200000 / a.loc[s.index, 'hours'].sum(), 3))).round(3)
by_staffing

In [ ]:
d = a[['nurse_hours_per_resident_day', 'handling_rate']].dropna()
r = stats.spearmanr(d.nurse_hours_per_resident_day, d.handling_rate)
print(f'nurse hours per resident-day vs staff handling-injury rate: rho = {r.statistic:.3f}, p = {r.pvalue:.4f}, n = {len(d):,}')

## 4. Two more joins, for illustration

In [ ]:
by_rating = a.dropna(subset=['overall_rating']).groupby('overall_rating').agg(
    facilities=('handling_rate', 'size'), median_handling_rate=('handling_rate', 'median'),
    pooled_handling_rate=('handling', lambda s: round(s.sum() * 200000 / a.loc[s.index, 'hours'].sum(), 3))).round(3)
by_ownership = a.groupby(a.ownership.str.split(' - ').str[0]).agg(
    facilities=('handling_rate', 'size'), median_handling_rate=('handling_rate', 'median')).round(3)
print(by_rating.to_string()); print(); print(by_ownership.to_string())

## What this example does not do

It is a demonstration of the join, not an analysis. Any real use would need to account for facility size, case mix, state, and the fact that an injury rate measures what was reported. Staffing, ratings, and injury rates are all measured at the facility level, so an association between them does not establish cause in either direction.

## Other things the published files support

- Join `hospital-crosswalk/osha-cms-crosswalk-hospital-2025.csv` to CMS hospital files the same way.
- Use `handling-task-types/derived/task-review-labels.csv` with its sample to score your own text classifier against a human-labeled benchmark.
- Use `benchmark/` (500 labeled narratives) to develop one.
- Use `trends-2023-2025/derived/trend-2023-2025.csv` for a three-year baseline.